# TalentLens
Extracción de información de CVs con PLN para agilizar la preselección de candidatos.

## 1. Captura de información
Leemos el texto de los CVs en PDF y Word. Los PDF escaneados se leen con OCR.

In [7]:
from pathlib import Path

import pymupdf
import docx
import pytesseract
import pandas as pd
from PIL import Image

DATA = Path('CVs_ficticios_prueba')
archivos = sorted(list(DATA.glob('pdf/*.pdf')) + list(DATA.glob('word/*.docx')))

print(len(archivos), 'CVs encontrados')
print('Tesseract', pytesseract.get_tesseract_version())
print('Idiomas OCR:', pytesseract.get_languages())

20 CVs encontrados
Tesseract 5.5.3
Idiomas OCR: ['afr', 'amh', 'ara', 'asm', 'aze', 'aze_cyrl', 'bel', 'ben', 'bod', 'bos', 'bre', 'bul', 'cat', 'ceb', 'ces', 'chi_sim', 'chi_sim_vert', 'chi_tra', 'chi_tra_vert', 'chr', 'cos', 'cym', 'dan', 'deu', 'div', 'dzo', 'ell', 'eng', 'enm', 'epo', 'equ', 'est', 'eus', 'fao', 'fas', 'fil', 'fin', 'fra', 'frk', 'frm', 'fry', 'gla', 'gle', 'glg', 'grc', 'guj', 'hat', 'heb', 'hin', 'hrv', 'hun', 'hye', 'iku', 'ind', 'isl', 'ita', 'ita_old', 'jav', 'jpn', 'jpn_vert', 'kan', 'kat', 'kat_old', 'kaz', 'khm', 'kir', 'kmr', 'kor', 'kor_vert', 'lao', 'lat', 'lav', 'lit', 'ltz', 'mal', 'mar', 'mkd', 'mlt', 'mon', 'mri', 'msa', 'mya', 'nep', 'nld', 'nor', 'oci', 'ori', 'osd', 'pan', 'pol', 'por', 'pus', 'que', 'ron', 'rus', 'san', 'sin', 'slk', 'slv', 'snd', 'spa', 'spa_old', 'sqi', 'srp', 'srp_latn', 'sun', 'swa', 'swe', 'syr', 'tam', 'tat', 'tel', 'tgk', 'tha', 'tir', 'ton', 'tur', 'uig', 'ukr', 'urd', 'uzb', 'uzb_cyrl', 'vie', 'yid', 'yor']


### Funciones de lectura
- **PDF:** texto con pymupdf; si la página no tiene texto (escaneada) se usa OCR en español (`--psm 4` mantiene juntas las filas de las tablas).
- **Word:** párrafos y tablas en el orden del documento.

In [12]:
def leer_pdf(ruta):
    """Texto de un PDF; si una página no tiene texto, se lee con OCR."""
    paginas = []
    with pymupdf.open(ruta) as pdf:
        for pagina in pdf:
            texto = pagina.get_text()
            if len(texto.strip()) < 20:
                pix = pagina.get_pixmap(dpi=300)
                img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
                texto = pytesseract.image_to_string(img, lang='spa', config='--psm 4')
            paginas.append(texto)
    return '\n'.join(paginas)

def leer_docx(ruta):
    """Texto de un Word en orden, incluyendo tablas."""
    doc = docx.Document(ruta)
    lineas = []
    for bloque in doc.iter_inner_content():
        if isinstance(bloque, docx.table.Table):
            for fila in bloque.rows:
                celdas = []
                for celda in fila.cells:
                    if celda.text not in celdas:  # celdas combinadas se repiten
                        celdas.append(celda.text)
                lineas.append('\n'.join(celdas))
        else:
            lineas.append(bloque.text)
    return '\n'.join(lineas)

def leer_cv(ruta):
    ruta = Path(ruta)
    if ruta.suffix.lower() == '.pdf':
        return leer_pdf(ruta)
    if ruta.suffix.lower() == '.docx':
        return leer_docx(ruta)
    raise ValueError(f'Formato no soportado: {ruta.suffix}')

### Leer todos los CVs

In [9]:
cvs = pd.DataFrame({
    'archivo': [str(a.relative_to(DATA)) for a in archivos],
    'texto': [leer_cv(a) for a in archivos],
})
cvs['caracteres'] = cvs['texto'].str.len()
cvs[['archivo', 'caracteres']]

,archivo,caracteres
0,pdf/cv_01_clasico.pdf,698
1,pdf/cv_02_moderno_foto.pdf,1052
2,pdf/cv_03_tabla.pdf,1060
3,pdf/cv_05_creativo_foto.pdf,840
4,pdf/cv_06_clasico.pdf,1205
5,pdf/cv_07_escaneado.pdf,888
6,pdf/cv_09_moderno_foto.pdf,753
7,pdf/cv_10_varias_paginas.pdf,2206
8,pdf/cv_11_creativo_foto.pdf,685
9,pdf/cv_13_tabla.pdf,1101


## 2. Limpieza
Decisiones:
- Normalizar unicode (NFC) y espacios.
- Quitar viñetas y unificar guiones largos.
- Unir correos partidos en dos líneas.
- Reparar la `@` mal leída por el OCR (solo dominios conocidos).
- Mantener mayúsculas: el NER las usa para detectar nombres.

In [13]:
import re
import unicodedata

VINETAS = '•·«»*▪●■◦–—-'
DOMINIOS = r'(gmail|yahoo|outlook|hotmail)\.com'

def limpiar_texto(texto):
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.replace('\xa0', ' ').replace('\t', ' ')
    # correos partidos en dos líneas (columnas angostas): "outlook.\ncom"
    texto = re.sub(r'(\S+@\S+)\n([a-z.]{1,4})$', r'\1\2', texto, flags=re.M)
    # OCR: la @ se lee como "0", "QW" o "GW" -> pedro.xicara890gmail.com
    texto = re.sub(r'\b([a-z][\w.]*?)[0OQGW]{1,2}' + DOMINIOS, r'\1@\2.com', texto)
    lineas = []
    for linea in texto.split('\n'):
        linea = linea.strip().lstrip(VINETAS).strip()  # viñetas al inicio
        linea = re.sub(r'\s*[–—]\s*', ' - ', linea)      # guiones largos
        linea = re.sub(r' {2,}', ' ', linea)
        if linea:
            lineas.append(linea)
    return '\n'.join(lineas)

In [14]:
cvs['limpio'] = cvs['texto'].map(limpiar_texto)
print(cvs.loc[1, 'limpio'][:400])

Contacto
+502 4260-9980
carlos.rodriguez86@outlook.com
Escuintla
Competencias
AWS
PostgreSQL
REST APIs
Python
Linux
Idiomas
Español (nativo)
K'iche' (intermedio)
Francés (básico)
Carlos Enrique Rodríguez López
Desarrollador Backend
Sobre mí
Desarrollador Backend con 9 años de experiencia. Me interesa aportar en equipos que
buscan resultados medibles y mejora continua.
Experiencia
Desarrollador Sen


### Separar secciones
Cada CV nombra sus secciones distinto (Experiencia / Trayectoria / Historial laboral, Skills / Aptitudes...). Las agrupamos con un diccionario de sinónimos.

Lo que está antes del primer encabezado queda como `encabezado`.

In [15]:
SECCIONES = {
    'contacto': ['contacto', 'datos personales', 'informacion de contacto'],
    'perfil': ['perfil', 'perfil profesional', 'sobre mi', 'resumen', 'objetivo', 'acerca de mi', 'profile'],
    'experiencia': ['experiencia', 'experiencia laboral', 'experiencia profesional', 'trayectoria',
                    'trayectoria profesional', 'historial laboral', 'experience'],
    'educacion': ['educacion', 'formacion', 'formacion academica', 'preparacion academica', 'estudios', 'education'],
    'habilidades': ['habilidades', 'competencias', 'skills', 'aptitudes', 'conocimientos',
                    'conocimientos tecnicos', 'herramientas y conocimientos'],
    'idiomas': ['idiomas', 'lenguas', 'languages'],
    'cursos': ['cursos', 'certificaciones', 'cursos y certificaciones'],
    'referencias': ['referencias'],
    'otros': ['informacion adicional'],
}
ENCABEZADOS = {frase: sec for sec, frases in SECCIONES.items() for frase in frases}
# segunda palabra de un encabezado partido en dos líneas (tablas): "EXPERIENCIA\nPROFESIONAL"
CONTINUACION = {'profesional', 'laboral', 'academica', 'tecnicos'}

def sin_acentos(s):
    s = unicodedata.normalize('NFD', s.lower())
    return ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')

def detectar_encabezado(linea):
    """Devuelve (sección, resto de la línea) o (None, línea)."""
    clave = sin_acentos(linea).strip(' :')
    if clave in ENCABEZADOS:
        return ENCABEZADOS[clave], ''
    # OCR de tablas: encabezado en mayúsculas pegado al contenido, "PERFIL Coordinador ..."
    palabras = linea.split()
    for n in (2, 1):
        inicio = ' '.join(palabras[:n])
        if inicio.isupper() and sin_acentos(inicio) in ENCABEZADOS and len(palabras) > n:
            return ENCABEZADOS[sin_acentos(inicio)], ' '.join(palabras[n:])
    return None, linea

def separar_secciones(texto):
    secciones = {'encabezado': []}
    actual = 'encabezado'
    anterior_fue_encabezado = False
    for linea in texto.split('\n'):
        primera = sin_acentos(linea.split()[0])
        if anterior_fue_encabezado and primera in CONTINUACION and linea.split()[0].isupper():
            linea = ' '.join(linea.split()[1:])
            if not linea:
                continue
        seccion, resto = detectar_encabezado(linea)
        anterior_fue_encabezado = seccion is not None
        if seccion:
            actual = seccion
            secciones.setdefault(actual, [])
            linea = resto
        if linea:
            secciones[actual].append(linea)
    return {sec: '\n'.join(lineas) for sec, lineas in secciones.items()}

In [16]:
cvs['secciones'] = cvs['limpio'].map(separar_secciones)

pd.DataFrame([{sec: '✓' for sec in s} for s in cvs['secciones']], index=cvs['archivo']).fillna('')

,encabezado,perfil,experiencia,educacion,habilidades,idiomas,contacto,cursos,referencias,otros
archivo,,,,,,,,,,
pdf/cv_01_clasico.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_02_moderno_foto.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_03_tabla.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_05_creativo_foto.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_06_clasico.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_07_escaneado.pdf,✓,✓,✓,✓,✓,✓,,,,
pdf/cv_09_moderno_foto.pdf,✓,✓,✓,✓,✓,✓,✓,,,
pdf/cv_10_varias_paginas.pdf,✓,✓,✓,✓,✓,✓,,✓,✓,✓
pdf/cv_11_creativo_foto.pdf,✓,✓,✓,✓,✓,✓,,,,


## 3. Extracción de información
### Datos de contacto y nombre
- **Correo y teléfono:** expresiones regulares.
- **Nombre:** NER preentrenado de spaCy (`es_core_news_sm`) + reglas. Se ignoran las secciones donde aparecen otras personas (referencias).

In [17]:
import spacy

nlp = spacy.load('es_core_news_sm')

RE_EMAIL = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')
RE_TELEFONO = re.compile(r'(?:\+502\s?)?\d{4}[-\s]?\d{4}')
CONECTORES = {'de', 'del', 'la', 'los', 'las', 'y'}

def extraer_email(texto):
    m = RE_EMAIL.search(texto)
    return m.group().rstrip('.') if m else None

def extraer_telefono(texto):
    m = RE_TELEFONO.search(texto)
    return m.group() if m else None

def parece_nombre(linea):
    palabras = linea.split()
    if not 2 <= len(palabras) <= 6 or any(ch.isdigit() for ch in linea):
        return False
    if detectar_encabezado(linea)[0]:
        return False
    return all(p[0].isupper() or p.lower() in CONECTORES for p in palabras)

def extraer_nombre(secciones):
    """Nombre del candidato: línea con forma de nombre que el NER marca como persona."""
    candidatos = []
    posicion = 0
    for seccion, texto in secciones.items():
        if seccion in ('referencias', 'experiencia', 'educacion', 'cursos'):
            continue  # aquí aparecen otras personas, empresas o títulos
        for linea in texto.split('\n'):
            posicion += 1
            if not parece_nombre(linea):
                continue
            doc = nlp(linea.title() if linea.isupper() else linea)
            es_persona = any(e.label_ == 'PER' and len(e.text) >= 0.8 * len(doc.text) for e in doc.ents)
            puntaje = 2 * es_persona + (len(linea.split()) >= 3) - 0.1 * posicion
            candidatos.append((puntaje, linea))
    if not candidatos:
        return None
    nombre = max(candidatos)[1]
    return nombre.title() if nombre.isupper() else nombre

In [18]:
cvs['nombre'] = cvs['secciones'].map(extraer_nombre)
cvs['email'] = cvs['limpio'].map(extraer_email)
cvs['telefono'] = cvs['limpio'].map(extraer_telefono)

cvs[['archivo', 'nombre', 'email', 'telefono']]

,archivo,nombre,email,telefono
0,pdf/cv_01_clasico.pdf,Ana Lucía Morales Pérez,ana.morales91@gmail.com,+502 4510-6956
1,pdf/cv_02_moderno_foto.pdf,Carlos Enrique Rodríguez López,carlos.rodriguez86@outlook.com,+502 4260-9980
2,pdf/cv_03_tabla.pdf,María José García Castillo,maria.garcia92@yahoo.com,+502 5110-5240
3,pdf/cv_05_creativo_foto.pdf,Sofía Isabel Méndez Orellana,sofia.mendez46@yahoo.com,+502 3105-1419
4,pdf/cv_06_clasico.pdf,Luis Fernando Ramírez Estrada,luis.ramirez18@yahoo.com,+502 3656-4047
5,pdf/cv_07_escaneado.pdf,Andrea Paola Cifuentes Monzón,NaN,+502 4353-7684
6,pdf/cv_09_moderno_foto.pdf,Gabriela Chávez Solares,gabriela.chavez72@yahoo.com,+502 5361-6739
7,pdf/cv_10_varias_paginas.pdf,Juan Pablo Barrios Girón,juan.barrios72@gmail.com,+502 3229-2844
8,pdf/cv_11_creativo_foto.pdf,Valeria Samayoa Reyes,valeria.samayoa19@outlook.com,+502 3934-1382
9,pdf/cv_13_tabla.pdf,Daniela Alejandra Ortiz Juárez,daniela.ortiz69@outlook.com,+502 5115-3589


### Años de experiencia
Se leen los rangos de fechas de la sección de experiencia, en cuatro formatos: `mar 2021`, `03/2021`, `Marzo de 2021` y `2021`.
- "actualidad", "a la fecha", "presente" y "actual" se toman como la fecha de referencia (sept 2026).
- Si solo aparece el año, se usa julio (mitad de año).
- Los periodos que se traslapan se cuentan una sola vez.

In [19]:
FECHA_REFERENCIA = (2026, 9)  # septiembre 2026, igual que el ground truth

MESES = {'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
         'jul': 7, 'ago': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12}
ACTUAL = r'(?:actualidad|a la fecha|presente|actual)'
# "mar 2021", "Marzo de 2021", "03/2021" o solo "2021"
FECHA = r'(?:[a-záéíóú]+(?: de)? \d{4}|\d{1,2}/\d{4}|\d{4})'
RE_RANGO = re.compile(rf'({FECHA})\s*-\s*({FECHA}|{ACTUAL})')

def leer_fecha(texto, es_fin=False):
    """Convierte una fecha a (año, mes). Si solo hay año, se usa mitad de año (julio)."""
    texto = texto.lower().strip()
    if re.fullmatch(ACTUAL, texto):
        return FECHA_REFERENCIA
    if m := re.fullmatch(r'(\d{1,2})/(\d{4})', texto):
        return int(m[2]), int(m[1])
    if m := re.fullmatch(r'([a-záéíóú]+)(?: de)? (\d{4})', texto):
        return int(m[2]), MESES.get(m[1][:3], 7)
    return int(texto), 7

def extraer_periodos(texto):
    periodos = []
    for inicio, fin in RE_RANGO.findall(texto.lower()):
        a, b = leer_fecha(inicio), min(leer_fecha(fin), FECHA_REFERENCIA)
        if a < b:
            periodos.append((a[0] * 12 + a[1], b[0] * 12 + b[1]))
    return periodos

def anios_experiencia(secciones):
    """Suma los periodos de la sección de experiencia sin contar dos veces los traslapes."""
    periodos = sorted(extraer_periodos(secciones.get('experiencia', '')))
    meses, fin_anterior = 0, 0
    for inicio, fin in periodos:
        inicio = max(inicio, fin_anterior)
        if fin > inicio:
            meses += fin - inicio
            fin_anterior = fin
    return round(meses / 12, 1)

In [20]:
cvs['anios_experiencia'] = cvs['secciones'].map(anios_experiencia)

cvs[['archivo', 'nombre', 'anios_experiencia']]

,archivo,nombre,anios_experiencia
0,pdf/cv_01_clasico.pdf,Ana Lucía Morales Pérez,1.3
1,pdf/cv_02_moderno_foto.pdf,Carlos Enrique Rodríguez López,9.8
2,pdf/cv_03_tabla.pdf,María José García Castillo,7.5
3,pdf/cv_05_creativo_foto.pdf,Sofía Isabel Méndez Orellana,4.6
4,pdf/cv_06_clasico.pdf,Luis Fernando Ramírez Estrada,8.8
5,pdf/cv_07_escaneado.pdf,Andrea Paola Cifuentes Monzón,5.7
6,pdf/cv_09_moderno_foto.pdf,Gabriela Chávez Solares,2.5
7,pdf/cv_10_varias_paginas.pdf,Juan Pablo Barrios Girón,7.8
8,pdf/cv_11_creativo_foto.pdf,Valeria Samayoa Reyes,1.5
9,pdf/cv_13_tabla.pdf,Daniela Alejandra Ortiz Juárez,6.2


### NER ajustado: empresas, educación y habilidades
Entrenamos un NER de spaCy con 4 etiquetas propias: `EMPRESA`, `TITULO`, `INSTITUCION` y `HABILIDAD`.

**Anotaciones:** no se etiquetó a mano. Cada valor del CSV de respuestas se busca en el texto del CV y se marca su posición.

In [21]:
gt = pd.read_csv(DATA / 'etiquetas_ground_truth.csv', encoding='utf-8-sig').set_index('archivo')

ETIQUETAS = {'empresas': 'EMPRESA', 'titulos_educacion': 'TITULO',
             'instituciones': 'INSTITUCION', 'habilidades': 'HABILIDAD'}

def anotar(texto, fila):
    """Busca en el texto cada valor del CSV y devuelve sus posiciones (inicio, fin, etiqueta)."""
    encontrados = []
    for columna, etiqueta in ETIQUETAS.items():
        for valor in set(str(fila[columna]).split('; ')):
            patron = r'\s+'.join(map(re.escape, valor.split()))  # tolera saltos de línea
            for m in re.finditer(rf'(?<!\w){patron}(?!\w)', texto, flags=re.I):
                encontrados.append((m.start(), m.end(), etiqueta))
    # si dos anotaciones se cruzan, gana la más larga
    encontrados.sort(key=lambda e: e[0] - e[1])
    spans = []
    for ini, fin, etq in encontrados:
        if all(fin <= a or ini >= b for a, b, _ in spans):
            spans.append((ini, fin, etq))
    return sorted(spans)

cvs['entidades'] = [anotar(t, gt.loc[a]) for a, t in zip(cvs['archivo'], cvs['limpio'])]

ejemplo = cvs.loc[0]
[(ejemplo['limpio'][i:f], e) for i, f, e in ejemplo['entidades']]

[('Agroexport del Pacífico', 'EMPRESA'),
 ('Python', 'HABILIDAD'),
 ('Power BI', 'HABILIDAD'),
 ('Licenciatura en Estadística', 'TITULO'),
 ('Universidad Francisco Marroquín', 'INSTITUCION'),
 ('Perito Contador', 'TITULO'),
 ('Instituto Nacional Central para Varones', 'INSTITUCION'),
 ('Pandas', 'HABILIDAD'),
 ('Tableau', 'HABILIDAD'),
 ('SQL', 'HABILIDAD'),
 ('Python', 'HABILIDAD'),
 ('R', 'HABILIDAD'),
 ('Estadística', 'HABILIDAD'),
 ('Power BI', 'HABILIDAD')]

**Entrenamiento y prueba:** 14 CVs para entrenar y 6 para evaluar, uno de cada tipo de formato (tabla, escaneado, dos columnas, creativo, Word con encabezados distintos y Word con tablas).

In [22]:
PRUEBA = ['cv_03', 'cv_07', 'cv_09', 'cv_11', 'cv_12', 'cv_17']
cvs['prueba'] = cvs['archivo'].str.contains('|'.join(PRUEBA))
cvs['prueba'].value_counts()

prueba
False    14
True      6
Name: count, dtype: int64

**Modelo:** se parte de un pipeline vacío en español (`spacy.blank('es')`). Probamos partir de `es_core_news_sm` y dio el mismo resultado, porque el NER de spaCy trae su propia capa de vectores.

In [23]:
import random
from spacy.training import Example

random.seed(0)
spacy.util.fix_random_seed(0)

ner_cv = spacy.blank('es')
ner = ner_cv.add_pipe('ner')
for etiqueta in ETIQUETAS.values():
    ner.add_label(etiqueta)

def a_ejemplos(filas):
    return [Example.from_dict(ner_cv.make_doc(t), {'entities': e})
            for t, e in zip(filas['limpio'], filas['entidades'])]

entrenamiento = a_ejemplos(cvs[~cvs['prueba']])
prueba = a_ejemplos(cvs[cvs['prueba']])

optimizador = ner_cv.initialize(lambda: entrenamiento)
for epoca in range(60):
    random.shuffle(entrenamiento)
    perdidas = {}
    for lote in spacy.util.minibatch(entrenamiento, size=4):
        ner_cv.update(lote, drop=0.2, losses=perdidas, sgd=optimizador)
    if (epoca + 1) % 10 == 0:
        f1 = ner_cv.evaluate(prueba)['ents_f']
        print(f'época {epoca + 1}: pérdida {perdidas["ner"]:.1f}, F1 prueba {f1:.3f}')

época 10: pérdida 485.8, F1 prueba 0.000
época 20: pérdida 329.7, F1 prueba 0.234
época 30: pérdida 174.5, F1 prueba 0.631
época 40: pérdida 15.8, F1 prueba 0.908
época 50: pérdida 5.3, F1 prueba 0.914
época 60: pérdida 5.4, F1 prueba 0.908


### Evaluación del NER (CVs de prueba)

In [24]:
resultado = ner_cv.evaluate(prueba)
print(f"Total -> P {resultado['ents_p']:.2f}  R {resultado['ents_r']:.2f}  F1 {resultado['ents_f']:.2f}")
pd.DataFrame(resultado['ents_per_type']).T.round(2)

Total -> P 0.87  R 0.95  F1 0.91


,p,r,f
TITULO,0.93,1.00,0.96
EMPRESA,0.79,0.92,0.85
HABILIDAD,0.84,0.91,0.88
INSTITUCION,1.00,1.00,1.00


Los CVs ficticios repiten muchas empresas y habilidades. Separamos el recall entre valores **vistos** en entrenamiento y valores **nuevos** para ver si el modelo generaliza o solo memoriza.

In [25]:
vistos = {(e.text.lower(), e.label_) for ej in entrenamiento for e in ej.reference.ents}
conteo = {'visto': [0, 0], 'nuevo': [0, 0]}  # [aciertos, total]

for ej in prueba:
    predichas = {(e.start_char, e.end_char, e.label_) for e in ner_cv(ej.reference.text).ents}
    for e in ej.reference.ents:
        tipo = 'visto' if (e.text.lower(), e.label_) in vistos else 'nuevo'
        conteo[tipo][0] += (e.start_char, e.end_char, e.label_) in predichas
        conteo[tipo][1] += 1

for tipo, (aciertos, total) in conteo.items():
    print(f'{tipo}: {aciertos}/{total} encontradas (recall {aciertos / total:.2f})')

visto: 67/69 encontradas (recall 0.97)
nuevo: 2/4 encontradas (recall 0.50)


In [26]:
from spacy import displacy

doc = ner_cv(cvs.loc[cvs['archivo'].str.contains('cv_12'), 'limpio'].item())
displacy.render(doc, style='ent', jupyter=True)

### Aplicar el NER a todos los CVs
Guardamos el modelo para usarlo en la app.

In [27]:
ner_cv.to_disk('modelo_ner')

def extraer_entidades(texto):
    doc = ner_cv(texto)
    campos = {etiqueta: [] for etiqueta in ETIQUETAS.values()}
    for e in doc.ents:
        valor = ' '.join(e.text.split())  # une saltos de línea
        if valor.lower() not in [v.lower() for v in campos[e.label_]]:
            campos[e.label_].append(valor)
    return {etiqueta: '; '.join(valores) for etiqueta, valores in campos.items()}

entidades = pd.DataFrame(cvs['limpio'].map(extraer_entidades).tolist())
cvs[['empresas', 'titulos', 'instituciones', 'habilidades']] = entidades[['EMPRESA', 'TITULO', 'INSTITUCION', 'HABILIDAD']].values

cvs[['nombre', 'empresas', 'titulos', 'habilidades']]

,nombre,empresas,titulos,habilidades
0,Ana Lucía Morales Pérez,Agroexport del Pacífico,Licenciatura en Estadística; Perito Contador,Python; Power BI; Pandas; Tableau; SQL; R; Est...
1,Carlos Enrique Rodríguez López,Distribuidora La Ceiba; Farmacias Bienestar; G...,Ingeniería en Ciencias y Sistemas; Perito Cont...,AWS; PostgreSQL; REST APIs; Python; Linux
2,María José García Castillo,Contadora Pública; Contadora Pública con; Audi...,María José García Castillo; Licenciatura en Co...,Seguros Horizonte; Conciliaciones bancarias; C...
3,Sofía Isabel Méndez Orellana,Consultores Asociados Ixchel; Almacenes El Faro,Licenciatura en Diseño Gráfico; Bachillerato e...,Figma; InDesign; Adobe Illustrator; Canva; Aft...
4,Luis Fernando Ramírez Estrada,Textiles San Juan; Energía Renovable Chiquimul...,Posgrado en Analítica de Datos; Licenciatura e...,Manejo de expedientes; Soporte vital básico; P...
5,Andrea Paola Cifuentes Monzón,Café Altiplano; TecnoCentro GT,Licenciatura en Mercadeo; Perito Contador,SEO; Google Analytics; Google Ads; Canva; HubS...
6,Gabriela Chávez Solares,Farmacias Bienestar,Licenciatura en Psicología Industrial; Perito ...,Reclutamiento y selección; Pruebas psicométric...
7,Juan Pablo Barrios Girón,Logística Maya; EduPlus Academia; Clínica Sant...,Licenciatura en Administración de Empresas; Pe...,importaciones; WMS; Gestión de inventarios; Ex...
8,Valeria Samayoa Reyes,Innova Digital,Ingeniería en Ciencias y Sistemas; Perito Cont...,REST APIs; PostgreSQL; Linux; AWS; Django; Doc...
9,Daniela Alejandra Ortiz Juárez,Energía Renovable Chiquimulilla; Hotel Posada ...,Ingeniería Industrial; Bachillerato en Ciencia...,SAP; Gestión de proyectos; Minitab; AutoCAD; E...


## 4. Evaluación contra el CSV de respuestas
### Datos de contacto y experiencia (20 CVs)
Estos campos usan reglas, no entrenamiento, así que se evalúan en todos los CVs.

In [28]:
def coincide(columna_pred, columna_real):
    reales = cvs['archivo'].map(gt[columna_real]).fillna('')
    return (cvs[columna_pred].fillna('') == reales).mean()

error_anios = (cvs['anios_experiencia'] - cvs['archivo'].map(gt['anios_experiencia'])).abs()

print(f"Nombre:   {coincide('nombre', 'nombre'):.0%} correctos")
print(f"Correo:   {coincide('email', 'email'):.0%} correctos")
print(f"Teléfono: {coincide('telefono', 'telefono'):.0%} correctos")
print(f"Años de experiencia: error promedio {error_anios.mean():.2f} años, exactos {(error_anios == 0).mean():.0%}")

Nombre:   100% correctos
Correo:   100% correctos
Teléfono: 100% correctos
Años de experiencia: error promedio 0.23 años, exactos 80%


### Entidades del NER (6 CVs de prueba)
Se compara la lista extraída con la lista del CSV, sin importar orden ni mayúsculas.

In [29]:
def a_conjunto(valor):
    if not isinstance(valor, str) or not valor.strip():
        return set()
    return {' '.join(v.lower().split()) for v in valor.split(';') if v.strip()}

def comparar(predicho, real):
    """Precisión y recall entre dos listas separadas por ';' (sin importar orden ni mayúsculas)."""
    p, r = a_conjunto(predicho), a_conjunto(real)
    return len(p & r), len(p), len(r)

def evaluar_campos(filas, campos):
    tabla = []
    for campo_pred, campo_real in campos.items():
        aciertos = predichos = reales = 0
        for _, fila in filas.iterrows():
            a, p, r = comparar(fila[campo_pred], gt.loc[fila['archivo'], campo_real])
            aciertos, predichos, reales = aciertos + a, predichos + p, reales + r
        precision = aciertos / predichos if predichos else 0
        recall = aciertos / reales if reales else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        tabla.append({'campo': campo_pred, 'precision': precision, 'recall': recall, 'f1': f1})
    return pd.DataFrame(tabla).set_index('campo').round(2)

CAMPOS = {'empresas': 'empresas', 'titulos': 'titulos_educacion',
          'instituciones': 'instituciones', 'habilidades': 'habilidades'}

evaluar_campos(cvs[cvs['prueba']], CAMPOS)

,precision,recall,f1
campo,,,
empresas,0.79,0.92,0.85
titulos,0.93,1.00,0.96
instituciones,1.00,1.00,1.00
habilidades,0.86,0.97,0.91


### Línea base: NER preentrenado sin ajustar
`es_core_news_sm` solo conoce etiquetas genéricas. Usamos `ORG` como empresa y `MISC` como habilidad.

In [30]:
def entidades_preentrenado(texto, etiqueta):
    valores = [' '.join(e.text.split()) for e in nlp(texto).ents if e.label_ == etiqueta]
    return '; '.join(dict.fromkeys(valores))

prueba_cvs = cvs[cvs['prueba']].copy()
prueba_cvs['empresas_base'] = prueba_cvs['limpio'].map(lambda t: entidades_preentrenado(t, 'ORG'))
prueba_cvs['habilidades_base'] = prueba_cvs['limpio'].map(lambda t: entidades_preentrenado(t, 'MISC'))

base = evaluar_campos(prueba_cvs, {'empresas_base': 'empresas', 'habilidades_base': 'habilidades'})
ajustado = evaluar_campos(prueba_cvs, {'empresas': 'empresas', 'habilidades': 'habilidades'})
pd.DataFrame({'F1 preentrenado': base['f1'].values, 'F1 ajustado': ajustado['f1'].values},
             index=['empresas', 'habilidades'])

,F1 preentrenado,F1 ajustado
empresas,0.06,0.85
habilidades,0.23,0.91
